In [1]:
from google.colab import drive
import pandas as pd

# Mount Google Drive
drive.mount('/content/drive')

# Load dataset
file_path = '/content/drive/MyDrive/sms_scam_detection_dataset_merged_with_lang.csv'
df = pd.read_csv(file_path)

# View basic info
df.head()


Mounted at /content/drive


<ipython-input-1-251af9c79ebf>:9: DtypeWarning: Columns (2,3,4,5,6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


,label,text,URL,EMAIL,PHONE,lang,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10
0,ham,Your opinion about me? 1. Over 2. Jada 3. Kusr...,No,No,No,en,NaN,NaN,NaN,NaN,NaN
1,ham,What's up? Do you want me to come online? If y...,No,No,No,en,NaN,NaN,NaN,NaN,NaN
2,ham,So u workin overtime nigpun?,No,No,No,en,NaN,NaN,NaN,NaN,NaN
3,ham,"Also sir, i sent you an email about how to log...",No,No,No,en,NaN,NaN,NaN,NaN,NaN
4,spam,Please Stay At Home. To encourage the notion o...,No,No,No,en,NaN,NaN,NaN,NaN,NaN


In [2]:
from google.colab import drive
import pandas as pd
import re

# Mount Google Drive
drive.mount('/content/drive')

# Load dataset
file_path = '/content/drive/MyDrive/sms_scam_detection_dataset_merged_with_lang.csv'
df = pd.read_csv(file_path)

# View basic info
df.head()

# Keep only required columns
df = df[['label', 'text']]

# Drop rows with missing text
df.dropna(subset=['text'], inplace=True)

# Normalize labels
df['label'] = df['label'].astype(str).str.strip().str.lower()
df['label'] = df['label'].map({'ham': 0, 'spam': 1})
df.dropna(subset=['label'], inplace=True)
df['label'] = df['label'].astype(int)

# Text cleaning function
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)
    text = re.sub(r'\@w+|\#','', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    return text.strip()

df['clean_text'] = df['text'].apply(clean_text)

# Check result
df.head()
df.to_csv("cleaned_sms_dataset.csv", index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<ipython-input-2-e2f634a13576>:10: DtypeWarning: Columns (2,3,4,5,6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


In [3]:
from google.colab import files
files.download("cleaned_sms_dataset.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
# Rule-based categorization
def categorize_message(text):
    text = text.lower()
    if any(word in text for word in ['otp', 'account', 'bank', 'transaction', 'loan', 'upi']):
        return 'bank'
    elif any(word in text for word in ['sale', 'free', 'offer', 'buy', 'discount', 'shopping', 'deal']):
        return 'shopping'
    elif any(word in text for word in ['win', 'prize', 'lottery', 'cashback', 'reward']):
        return 'promotion'
    elif any(word in text for word in ['meet', 'call', 'dinner', 'lunch', 'party', 'movie']):
        return 'personal'
    else:
        return 'others'

df['category'] = df['clean_text'].apply(categorize_message)
df[['label', 'text', 'category']].head()


,label,text,category
0,0,Your opinion about me? 1. Over 2. Jada 3. Kusr...,others
1,0,What's up? Do you want me to come online? If y...,shopping
2,0,So u workin overtime nigpun?,others
3,0,"Also sir, i sent you an email about how to log...",others
4,1,Please Stay At Home. To encourage the notion o...,others


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label'], test_size=0.2, random_state=42
)

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


In [6]:
from sklearn.linear_model import LogisticRegression

# Train logistic regression model
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)


LogisticRegression()

In [7]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Predict and evaluate
y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.7779418650722184
Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.83      0.81     15766
           1       0.76      0.71      0.73     11997

    accuracy                           0.78     27763
   macro avg       0.78      0.77      0.77     27763
weighted avg       0.78      0.78      0.78     27763

Confusion Matrix:
 [[13133  2633]
 [ 3532  8465]]


In [9]:
# Clean and predict new message
def test_message(text):
    cleaned = clean_text(text)
    vect = vectorizer.transform([cleaned])
    pred = model.predict(vect)[0]
    category = categorize_message(cleaned)
    return 'SPAM' if pred == 1 else 'HAM', category

# Example
msg = "Get 50% off, sales are limited!"
result, category = test_message(msg)
print(f"🔍 Message: {msg}\n✅ Classified as: {result}")


🔍 Message: Get 50% off, sales are limited!
✅ Classified as: HAM


In [ ]:
import joblib
joblib.dump(model, 'spam_model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')


['vectorizer.pkl']

In [10]:
# Detect spam message only
def is_spam_message(text):
    cleaned = clean_text(text)
    vect = vectorizer.transform([cleaned])
    pred = model.predict(vect)[0]
    return pred == 1  # Returns True if spam

# Example usage
msg = "Congratulations! You've won a free cruise. Call now!"
if is_spam_message(msg):
    print(f"🚨 SPAM detected:\n📩 Message: {msg}")
else:
    print("✅ Message is clean (not spam).")


🚨 SPAM detected:
📩 Message: Congratulations! You've won a free cruise. Call now!
